## 0. Split Dataset 8:2 - train:test

In [2]:
import zarr
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import CosineAnnealingLR
import os
import json
from tqdm import tqdm
import matplotlib.pyplot as plt

from lerobot.configs.types import FeatureType, PolicyFeature
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

DATASET_PATH = "/home/thanh-huynhbao/git/control_policy/dataset/pusht/pusht_cchi_v7_replay.zarr"
OUTPUT_PATH = "/home/thanh-huynhbao/git/control_policy/tutorials/lerobot_implement/outputs/train_diffusion_pusht_local"
CHECKPOINT_DIR = OUTPUT_PATH
CHECKPOINT_NAME = "checkpoint_last.pt"
OUTPUT_PLOT = os.path.join(OUTPUT_PATH, "diffusion_test_inference_eval.png")
# HyperParameters (aligned with official Diffusion Policy community standards)
SEED = 42
N_OBS_STEPS = 2        # Number of historical observations conditioned on (t-1, t)
HORIZON = 16           # Action prediction sequence length (must be a multiple of 8)
N_ACTION_STEPS = 8     # Actions executed per rollout
BATCH_SIZE = 64
NUM_WORKERS = 8
LR = 1e-4
WEIGHT_DECAY = 1e-6
EPOCH = 40
NUM_TRAIN_TIMESTEPS = 100
NUM_INFERENCE_STEPS = 10  # 10-step DDIM/DDPM: 10x faster inference with identical quality


/home/thanh-huynhbao/git/control_policy/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data-driven

In [3]:
class PushTDataset(Dataset):
    def __init__(
        self,
        zarr_path: str,
        episode_indices: list[int],
        horizon: int = 16,
        n_obs_steps: int = 2,
        stats: dict | None = None,
    ):
        self.horizon = horizon
        self.n_obs_steps = n_obs_steps
        self.episode_indices = episode_indices

        store = zarr.open(zarr_path, mode="r")
        all_actions = store["data"]["action"][:]  # (N, 2)
        all_states = store["data"]["state"][:]    # (N, 5)
        all_images = store["data"]["img"][:]      # (N, 96, 96, 3)
        all_episode_ends = store["meta"]["episode_ends"][:]

        # Build list of samples: (global_t, ep_start, ep_end)
        self.samples = []
        for episode_idx in episode_indices:
            episode_start = 0 if episode_idx == 0 else all_episode_ends[episode_idx - 1]
            episode_end = all_episode_ends[episode_idx]
            for t in range(episode_start, episode_end):
                self.samples.append((t, episode_start, episode_end))

        self.raw_actions = all_actions
        self.raw_states = all_states
        self.raw_images = all_images

        # Compute Min-Max statistics strictly on train set for Diffusion Policy [-1, 1] space
        if stats is None:
            train_states = np.concatenate(
                [
                    all_states[(0 if episode == 0 else all_episode_ends[episode - 1]) : all_episode_ends[episode]] for episode in episode_indices
                ], axis=0,
            )
            train_actions = np.concatenate(
                [
                    all_actions[(0 if episode == 0 else all_episode_ends[episode - 1]) : all_episode_ends[episode]] for episode in episode_indices
                ], axis=0,
            )

            self.stats = {
                "state_min": train_states.min(axis=0).tolist(),
                "state_max": train_states.max(axis=0).tolist(),
                "action_min": train_actions.min(axis=0).tolist(),
                "action_max": train_actions.max(axis=0).tolist(),
            }
        else:
            self.stats = stats

        self.state_min = np.array(self.stats["state_min"], dtype=np.float32)
        self.state_max = np.array(self.stats["state_max"], dtype=np.float32)
        self.action_min = np.array(self.stats["action_min"], dtype=np.float32)
        self.action_max = np.array(self.stats["action_max"], dtype=np.float32)

        self.state_denom = np.clip(self.state_max - self.state_min, a_min=1e-5, a_max=None)
        self.action_denom = np.clip(self.action_max - self.action_min, a_min=1e-5, a_max=None)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        t, episode_start, episode_end = self.samples[idx]

        # 1. Observation sequence: indices [t - n_obs_steps + 1, ..., t]
        obs_indices = [max(episode_start, t - i) for i in reversed(range(self.n_obs_steps))]

        # Stack images: (n_obs_steps, 3, 96, 96) scaled to [0, 1]
        raw_imgs = self.raw_images[obs_indices].astype(np.float32) / 255.0
        imgs = np.transpose(raw_imgs, (0, 3, 1, 2))

        # Stack states: Min-Max normalized to [-1, 1]
        raw_states = self.raw_states[obs_indices].astype(np.float32)
        states = 2.0 * (raw_states - self.state_min) / self.state_denom - 1.0

        # 2. Action trajectory: indices starting from (t - n_obs_steps + 1)
        action_start = t - (self.n_obs_steps - 1)
        action_end = action_start + self.horizon

        actions_chunk = np.zeros((self.horizon, 2), dtype=np.float32)
        action_is_pad = np.zeros((self.horizon,), dtype=bool)

        for i, step in enumerate(range(action_start, action_end)):
            if step < episode_start:
                raw_act = self.raw_actions[episode_start].astype(np.float32)
            elif step < episode_end:
                raw_act = self.raw_actions[step].astype(np.float32)
            else:
                raw_act = self.raw_actions[episode_end - 1].astype(np.float32)
                action_is_pad[i] = True
            # Normalize action to [-1, 1]
            actions_chunk[i] = 2.0 * (raw_act - self.action_min) / self.action_denom - 1.0

        return {
            "observation.image": torch.from_numpy(imgs),
            "observation.state": torch.from_numpy(states),
            "action": torch.from_numpy(actions_chunk),
            "action_is_pad": torch.from_numpy(action_is_pad),
        }

def split_episodes(total_episodes: int, train_ratio: float=0.8, seed: int = 42):
    """Split episode indices into train and test sets."""
    np.random.seed(seed)
    shuffled_indices = np.random.permutation(total_episodes).tolist()
    n_train = int(total_episodes * train_ratio)
    train_indices = sorted(shuffled_indices[:n_train])
    test_indices = sorted(shuffled_indices[n_train:])
    return train_indices, test_indices


In [5]:

# 1. Inspect episodes and split 8:2
store = zarr.open(DATASET_PATH, mode="r")
episode_ends = store["meta"]["episode_ends"][:]
total_episodes = len(episode_ends)
train_indices, test_indices = split_episodes(total_episodes, train_ratio=0.8, seed=SEED)
print(f"Total episodes: {total_episodes}")
print(f"Train episodes: {len(train_indices)} ({len(train_indices)/total_episodes*100:.1f}%)")
print(f"Test episodes: {len(test_indices)} ({len(test_indices)/total_episodes*100:.1f}%)")

# 2. Build datasets
train_dataset = PushTDataset(DATASET_PATH, train_indices, horizon=HORIZON, n_obs_steps=N_OBS_STEPS)
test_dataset = PushTDataset(DATASET_PATH, test_indices, horizon=HORIZON, n_obs_steps=N_OBS_STEPS, stats=train_dataset.stats)
# 2. Data loader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Train transitions: {len(train_dataset)}, Batches: {len(train_loader)}")
print(f"Test transitions:  {len(test_dataset)}, Batches: {len(test_loader)}") 


Total episodes: 206
Train episodes: 164 (79.6%)
Test episodes: 42 (20.4%)
Train transitions: 20353, Batches: 318
Test transitions:  5297, Batches: 83


## 2. Model Prepare

In [7]:
# Select Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Configure LeRobot Diffusion Policy (community standard configuration)
config = DiffusionConfig(
    input_features={
        "observation.image": PolicyFeature(type=FeatureType.VISUAL, shape=(3, 96, 96)),
        "observation.state": PolicyFeature(type=FeatureType.STATE, shape=(5,)),
    },
    output_features={
        "action": PolicyFeature(type=FeatureType.ACTION, shape=(2,)),
    },
    n_obs_steps=N_OBS_STEPS,
    horizon=HORIZON,
    n_action_steps=N_ACTION_STEPS,
    vision_backbone="resnet18",
    crop_shape=None,
    crop_ratio=1.0,
    spatial_softmax_num_keypoints=32,
    down_dims=(256, 512, 1024),  # Standard 1D Temporal U-Net dimensions
    kernel_size=5,
    n_groups=8,
    diffusion_step_embed_dim=128,
    use_film_scale_modulation=True,
    noise_scheduler_type="DDPM",
    num_train_timesteps=NUM_TRAIN_TIMESTEPS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon",
    clip_sample=True,
    clip_sample_range=1.0,
    optimizer_lr=LR,
    optimizer_weight_decay=WEIGHT_DECAY,
)

policy = DiffusionPolicy(config).to(device)

# 4. Optimizer & Cosine Annealing Scheduler with Warmup
optimizer = torch.optim.AdamW(
    policy.get_optim_params(),
    lr=LR,
    betas=(0.95, 0.999),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)

total_steps = EPOCH * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)

# 5. Save metadata
meta_info = {
    "train_indices": train_indices,
    "test_indices": test_indices,
    "horizon": HORIZON,
    "n_obs_steps": N_OBS_STEPS,
    "n_action_steps": N_ACTION_STEPS,
    "stats": train_dataset.stats,
}

os.makedirs(OUTPUT_PATH, exist_ok=True)
with open(os.path.join(OUTPUT_PATH, "dataset_meta.json"), "w") as f:
    json.dump(meta_info, f, indent=2)
    print("Metadata saved to dataset_meta.json")


Using device: cuda
Metadata saved to dataset_meta.json


## 3. Train

In [8]:
print("\nStarting Diffusion Policy Training...")

train_loss_list = []
val_loss_list = []
best_val_loss = float("inf")

for epoch in range(1, EPOCH + 1):
    policy.train()
    train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCH} [Train]")
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        loss, _ = policy(batch)  # DiffusionPolicy returns (loss, None)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{scheduler.get_last_lr()[0]:.6f}"})

    avg_train_loss = train_loss / len(train_loader)
    train_loss_list.append(avg_train_loss)

    # Evaluate validation loss every 5 epochs and at final epoch
    if epoch % 5 == 0 or epoch == EPOCH:
        policy.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                v_loss, _ = policy(batch)
                val_loss += v_loss.item()
        avg_val_loss = val_loss / len(test_loader)
        val_loss_list.append(avg_val_loss)
        print(f"Epoch {epoch:02d} Summary: Train MSE: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f}")

        # Save best model checkpoint
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            checkpoint_best = {
                "epoch": epoch,
                "policy_state_dict": policy.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": config,
                "val_loss": avg_val_loss,
            }
            torch.save(checkpoint_best, os.path.join(OUTPUT_PATH, "checkpoint_best.pt"))
            print(f"[*] New best validation loss: {best_val_loss:.4f} - Saved checkpoint_best.pt")
    else:
        print(f"Epoch {epoch:02d} Summary: Train MSE: {avg_train_loss:.4f}")



Starting Diffusion Policy Training...


Epoch 1/40 [Train]: 100%|██████████| 318/318 [00:44<00:00,  7.09it/s, loss=0.0777, lr=0.000100]


Epoch 01 Summary: Train MSE: 0.1735


Epoch 2/40 [Train]: 100%|██████████| 318/318 [00:43<00:00,  7.23it/s, loss=0.0583, lr=0.000099]


Epoch 02 Summary: Train MSE: 0.0606


...

Epoch 38 Summary: Train MSE: 0.0167


Epoch 39/40 [Train]: 100%|██████████| 318/318 [00:43<00:00,  7.24it/s, loss=0.0093, lr=0.000001]


Epoch 39 Summary: Train MSE: 0.0162


Epoch 40/40 [Train]: 100%|██████████| 318/318 [00:43<00:00,  7.24it/s, loss=0.0103, lr=0.000001]


Epoch 40 Summary: Train MSE: 0.0164 | Val MSE: 0.0565


In [9]:
checkpoint_last = {
    "epoch": epoch,
    "policy_state_dict": policy.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": config,
    "train_loss": avg_train_loss,
}
torch.save(checkpoint_last, os.path.join(OUTPUT_PATH, "checkpoint_last.pt"))
print("Saved checkpoint_last.pt to", OUTPUT_PATH)


Saved checkpoint_last.pt to /home/thanh-huynhbao/git/control_policy/tutorials/lerobot_implement/outputs/train_diffusion_pusht_local


## 4. Inference with testset

In [5]:
EVAL_EPISODE = 1

def load_model_and_metadata(checkpoint_path: str, meta_path: str, device: torch.device):
    """Load trained DiffusionPolicy, configuration, and dataset metadata."""
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"Metadata not found: {meta_path}")

    with open(meta_path, "r") as f:
        meta_info = json.load(f)

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = checkpoint["config"]

    policy = DiffusionPolicy(config).to(device)
    policy.load_state_dict(checkpoint["policy_state_dict"])
    policy.num_inference_steps = getattr(config, "num_inference_steps", None) or 10
    policy.eval()

    return policy, config, meta_info

def run_episode_inference(
    policy: DiffusionPolicy,
    episode_images: np.ndarray,
    episode_states: np.ndarray,
    stats: dict,
    device: torch.device,
) -> np.ndarray:
    """
    Run closed-loop / receding horizon rollout on an episode.
    Feeds single observation steps to policy.select_action(batch),
    which internally manages the 2-step history queue and 8-step execution queue.
    """
    state_min = np.array(stats["state_min"], dtype=np.float32)
    state_max = np.array(stats["state_max"], dtype=np.float32)
    action_min = np.array(stats["action_min"], dtype=np.float32)
    action_max = np.array(stats["action_max"], dtype=np.float32)

    state_denom = np.clip(state_max - state_min, a_min=1e-5, a_max=None)
    action_range = action_max - action_min

    T = len(episode_images)
    predicted_actions = []

    # Reset policy queues per episode
    policy.reset()

    with torch.no_grad():
        for t in range(T):
            # 1. Normalize current observation frame to [0, 1] for images, [-1, 1] for states
            img = episode_images[t].astype(np.float32) / 255.0
            img = np.transpose(img, (2, 0, 1))
            img_tensor = torch.from_numpy(img).unsqueeze(0).to(device)  # (1, 3, 96, 96)

            norm_state = 2.0 * (episode_states[t].astype(np.float32) - state_min) / state_denom - 1.0
            state_tensor = torch.from_numpy(norm_state).unsqueeze(0).to(device)  # (1, 5)

            batch = {
                "observation.image": img_tensor,
                "observation.state": state_tensor,
            }

            # 2. Select action: internal queue manages 2-step history and 8-step receding horizon execution
            pred_norm_action = policy.select_action(batch)
            pred_norm_action = pred_norm_action.cpu().numpy().squeeze(0)

            # 3. Unnormalize [-1, 1] back to raw action pixel space [0, 512]
            pred_action = (pred_norm_action + 1.0) / 2.0 * action_range + action_min
            predicted_actions.append(pred_action)

    return np.array(predicted_actions)

def compute_metrics(ground_truth: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    """Compute MSE, RMSE, MAE, and per-axis errors between GT and predictions."""
    diff = predicted - ground_truth
    mse = np.mean(diff ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(diff))
    max_err = np.max(np.abs(diff))

    mae_x = np.mean(np.abs(diff[:, 0]))
    mae_y = np.mean(np.abs(diff[:, 1]))

    return {
        "MSE": float(mse),
        "RMSE": float(rmse),
        "MAE": float(mae),
        "MAE_X": float(mae_x),
        "MAE_Y": float(mae_y),
        "Max_Error": float(max_err),
    }

def plot_comparison(
    gt_actions: np.ndarray,
    pred_actions: np.ndarray,
    episode_idx: int,
    metrics: dict[str, float],
    output_path: str,
    action_gt: np.ndarray,
    action_pred: np.ndarray
):
    """Plot 2D trajectory and time-series comparison between GT and predicted actions."""
    fig = plt.figure(figsize=(15, 12))

    # Subplot 1: 2D Trajectory
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.plot(gt_actions[:, 0], gt_actions[:, 1], "b-", linewidth=2.0, label="Ground Truth", alpha=0.8)
    ax1.plot(pred_actions[:, 0], pred_actions[:, 1], "r--", linewidth=2.0, label="ACT Prediction", alpha=0.8)

    # Mark Start and End
    ax1.scatter(gt_actions[0, 0], gt_actions[0, 1], c="green", s=90, marker="o", label="Start", zorder=5)
    ax1.scatter(gt_actions[-1, 0], gt_actions[-1, 1], c="blue", s=90, marker="x", label="GT End", zorder=5)
    ax1.scatter(pred_actions[-1, 0], pred_actions[-1, 1], c="red", s=90, marker="^", label="Pred End", zorder=5)

    ax1.set_title(f"Test Episode {episode_idx}: 2D Action Trajectory", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Action Dim 0 (X coordinate)", fontsize=10)
    ax1.set_ylabel("Action Dim 1 (Y coordinate)", fontsize=10)
    ax1.legend(loc="best", fontsize=9)
    ax1.grid(True, linestyle="--", alpha=0.6)

    # Subplot 2: Time Series for X and Y
    ax2 = fig.add_subplot(4, 2, 2)
    ax2.plot(gt_actions[:, 0], "b-", label="GT X", linewidth=1.8)
    ax2.plot(pred_actions[:, 0], "r--", label="Pred X", linewidth=1.8)
    ax2.set_title(f"Action Dim 0 (X) [MAE: {metrics['MAE_X']:.2f}]", fontsize=11, fontweight="bold")
    ax2.set_ylabel("Position X", fontsize=9)
    ax2.legend(loc="upper right", fontsize=8)
    ax2.grid(True, linestyle="--", alpha=0.6)

    ax3 = fig.add_subplot(4, 2, 4)
    ax3.plot(gt_actions[:, 1], "b-", label="GT Y", linewidth=1.8)
    ax3.plot(pred_actions[:, 1], "r--", label="Pred Y", linewidth=1.8)
    ax3.set_title(f"Action Dim 1 (Y) [MAE: {metrics['MAE_Y']:.2f}]", fontsize=11, fontweight="bold")
    ax3.set_xlabel("Time Step", fontsize=10)
    ax3.set_ylabel("Position Y", fontsize=9)
    ax3.legend(loc="upper right", fontsize=8)
    ax3.grid(True, linestyle="--", alpha=0.6)

    # Subplot 2: Time Series for X and Y
    ax4 = fig.add_subplot(4, 1, 3)
    ax4.plot(action_gt[:, 0], "b-", label="GT X", linewidth=1.8)
    ax4.plot(action_pred[:, 0], "r--", label="Pred X", linewidth=1.8)
    ax4.set_title(f"Action Dim 0 (X) [MAE: {metrics['MAE_X']:.2f}]", fontsize=11, fontweight="bold")
    ax4.set_ylabel("Position X", fontsize=9)
    ax4.legend(loc="upper right", fontsize=8)
    ax4.grid(True, linestyle="--", alpha=0.6)

    ax5 = fig.add_subplot(4, 1, 4)
    ax5.plot(action_gt[:, 1], "b-", label="GT Y", linewidth=1.8)
    ax5.plot(action_pred[:, 1], "r--", label="Pred Y", linewidth=1.8)
    ax5.set_title(f"Action Dim 1 (Y) [MAE: {metrics['MAE_Y']:.2f}]", fontsize=11, fontweight="bold")
    ax5.set_xlabel("Time Step", fontsize=10)
    ax5.set_ylabel("Position Y", fontsize=9)
    ax5.legend(loc="upper right", fontsize=8)
    ax5.grid(True, linestyle="--", alpha=0.6)

    plt.suptitle(
        f"ACT Push-T Evaluation (Episode {episode_idx}) | Overall MAE: {metrics['MAE']:.2f}, RMSE: {metrics['RMSE']:.2f}",
        fontsize=13,
        fontweight="bold",
        y=0.98,
    )
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved inference visualization plot to: {output_path}")


In [6]:
# 1. Load model and metadata
checkpoint_to_load = os.path.join(CHECKPOINT_DIR, "checkpoint_best.pt")
if not os.path.exists(checkpoint_to_load):
    checkpoint_to_load = os.path.join(CHECKPOINT_DIR, CHECKPOINT_NAME)
meta_path = os.path.join(CHECKPOINT_DIR, "dataset_meta.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

policy, config, meta_info = load_model_and_metadata(checkpoint_to_load, meta_path, device)
test_indices = meta_info["test_indices"]
stats = meta_info["stats"]

print(f"Loaded checkpoint: {checkpoint_to_load}")
print(f"Test episodes count: {len(test_indices)} (indices: {test_indices[:5]} ...)")

# 2. Open dataset
store = zarr.open(DATASET_PATH, mode="r")
actions = store["data"]["action"][:]
states = store["data"]["state"][:]
images = store["data"]["img"][:]
episode_ends = store["meta"]["episode_ends"][:]

# 3. Evaluate test set
all_metrics = []
plot_gt, plot_pred, plot_metrics = None, None, None
target_plot_ep = EVAL_EPISODE if EVAL_EPISODE is not None else test_indices[0]
action_gt = np.empty((0,2))
action_pred = np.empty((0,2))
print(f"Running inference across {len(test_indices)} test episodes...")
for episode_idx in tqdm(test_indices, desc="Evaluating Test Episodes"):
    episode_start = 0 if episode_idx == 0 else episode_ends[episode_idx - 1]
    episode_end = episode_ends[episode_idx]

    episode_gt_actions = actions[episode_start:episode_end]
    episode_states = states[episode_start:episode_end]
    episode_images = images[episode_start:episode_end]

    action_gt = np.vstack([action_gt, episode_gt_actions])

    episode_predicted_actions = run_episode_inference(policy, episode_images, episode_states, stats, device)
    episode_metrics = compute_metrics(episode_gt_actions, episode_predicted_actions)
    all_metrics.append(episode_metrics)

    action_pred = np.vstack([action_pred, episode_predicted_actions])

    if episode_idx == target_plot_ep:
        plot_gt = episode_gt_actions
        plot_pred = episode_predicted_actions
        plot_metrics = episode_metrics


Using device: cuda
Loaded checkpoint: /home/thanh-huynhbao/git/control_policy/tutorials/lerobot_implement/outputs/train_diffusion_pusht_local/checkpoint_best.pt
Test episodes count: 42 (indices: [1, 14, 20, 21, 37] ...)
Running inference across 42 test episodes...


Evaluating Test Episodes: 100%|██████████| 42/42 [00:34<00:00,  1.21it/s]


## 5. Plot and evaluate

In [7]:
if plot_gt is not None and plot_pred is not None:
    plot_comparison(plot_gt, plot_pred, target_plot_ep, plot_metrics, OUTPUT_PLOT, action_gt, action_pred)


Saved inference visualization plot to: /home/thanh-huynhbao/git/control_policy/tutorials/lerobot_implement/outputs/train_diffusion_pusht_local/diffusion_test_inference_eval.png


In [13]:
# 4. Aggregate metrics 
mean_mse = np.mean([m["MSE"] for m in all_metrics])
mean_rmse = np.mean([m["RMSE"] for m in all_metrics])
mean_mae = np.mean([m["MAE"] for m in all_metrics])
mean_mae_x = np.mean([m["MAE_X"] for m in all_metrics])
mean_mae_y = np.mean([m["MAE_Y"] for m in all_metrics])
max_err = np.max([m["Max_Error"] for m in all_metrics])
print("\n" + "=" * 55)
print("           TEST SET EVALUATION METRICS SUMMARY           ")
print("=" * 55)
print(f"Evaluated Test Episodes: {len(test_indices)}")
print(f"Mean Squared Error (MSE):      {mean_mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {mean_rmse:.4f}")
print(f"Mean Absolute Error (MAE):     {mean_mae:.4f}")
print(f"  - Action X MAE:              {mean_mae_x:.4f}")
print(f"  - Action Y MAE:              {mean_mae_y:.4f}")
print(f"Max Absolute Error:            {max_err:.4f}")
print("=" * 55)


           TEST SET EVALUATION METRICS SUMMARY           
Evaluated Test Episodes: 42
Mean Squared Error (MSE):      1258.6968
Root Mean Squared Error (RMSE): 34.0084
Mean Absolute Error (MAE):     22.7545
  - Action X MAE:              21.7693
  - Action Y MAE:              23.7396
Max Absolute Error:            278.3106


In [6]:
import gc
import torch

def clear_gpu_memory():
    print(f"Allocated before: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Cached before:    {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    
    gc.collect()
    torch.cuda.empty_cache()
    
    print("--- Memory Cleared ---")
    print(f"Allocated after:  {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Cached after:     {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# Call the function
clear_gpu_memory()

Allocated before: 4.26 GB
Cached before:    5.69 GB
--- Memory Cleared ---
Allocated after:  4.26 GB
Cached after:     4.76 GB
